# Adversarial training

This code implements adversarial training using Projected Gradient Descent (PGD) on the CIFAR-10 dataset with a Convolutional Neural Network (CNN) in TensorFlow/Keras. The goal is to train a model that is more robust against adversarial attacks.

We import our required libraries and preprocess the data. Adversarial training is very computationally expensive (especially in a Python notebook) so we use a smaller dataset - CIFAR-10 is a dataset of 32x32 colour images categorized into 10 classes. We also further reduce it in size to save memory and speed up training. This snippet prepares a CIFAR-10 setup suitable for adversarial training experiments. It imports TensorFlow/Keras utilities, loads the CIFAR-10 dataset (32×32 RGB images across 10 classes), and then deliberately subsamples it to 10,000 training and 2,000 test examples to keep demos fast—useful when repeatedly generating adversarial examples during training. Images are cast to float32 and normalised to [0,1] by dividing by 255 for stable optimisation, and labels are one-hot encoded with to_categorical so the model can learn a probability distribution over the 10 classes. With this compact, preprocessed dataset in place, you can plug in a CNN and an attack routine (e.g., FGSM/PGD) to perform adversarial training efficiently.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load and preprocess CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Reduce dataset size for demonstration
x_train, y_train = x_train[:10000], y_train[:10000]
x_test, y_test = x_test[:2000], y_test[:2000]

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 792s 5us/step


Here you define a simple CNN architecture tailored for the CIFAR-10 dataset. The model uses two convolutional layers—first with 32 filters, then 64 filters—each followed by max pooling to reduce spatial dimensions while retaining important features. After flattening the extracted feature maps, the network passes them through a dense hidden layer of 64 ReLU-activated neurons, before outputting predictions through a 10-unit softmax layer (one per CIFAR-10 class). The model is compiled with the Adam optimiser, categorical cross-entropy loss, and accuracy as the metric, making it well-suited for multi-class image classification. This compact design is efficient enough for adversarial training experiments while still expressive enough to capture CIFAR-10 patterns.

In [2]:
# Define a simple CNN for CIFAR-10
def create_simple_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

This function implements a Projected Gradient Descent (PGD) attack, one of the strongest and most widely used adversarial attack methods. Starting with a copy of the original images, the algorithm iteratively perturbs them by moving in the direction of the gradient of the loss with respect to the input. Each step adds a small change (alpha) proportional to the sign of the gradient, nudging the image toward higher loss and misclassification. To keep the perturbation bounded, the updated images are clipped within an epsilon-ball around the originals (images ± epsilon), ensuring they remain close enough to look visually unchanged. Finally, the images are clipped again to [0,1] to maintain valid pixel values. Repeating this process for a set number of iterations produces adversarial examples that are subtle but highly effective at fooling the model.

In [3]:
# PGD attack implementation
def pgd_attack(model, images, labels, epsilon=0.03, alpha=0.007, iters=5):
    adv_images = tf.identity(images)
    for _ in range(iters):
        with tf.GradientTape() as tape:
            tape.watch(adv_images)
            predictions = model(adv_images)
            loss = tf.keras.losses.categorical_crossentropy(labels, predictions)
        gradients = tape.gradient(loss, adv_images)
        adv_images = adv_images + alpha * tf.sign(gradients)
        adv_images = tf.clip_by_value(adv_images, images - epsilon, images + epsilon)
        adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)
    return adv_images

This code performs on-the-fly adversarial training and optional adversarial evaluation. In each epoch, it streams through the training set in small batches to save memory, first crafting PGD adversarial examples (adv_x_batch = pgd_attack(...)) against the current model, then updating the model using only those adversarial inputs (train_on_batch). Training exclusively on adversarial samples hardens the decision boundary (though it can trade off some clean accuracy), and doing the attack inside the loop yields stronger, adaptive examples than precomputing once. The evaluate_model helper can test either on clean data or on a subset of adversarially perturbed test images (first 1,000) to estimate robustness. Note that when adversarial=False it still slices y_test[:1000], so you should also slice x_test[:1000] (or don’t slice either) to keep lengths aligned.

In [4]:
# Adversarial training function
def adversarial_training(model, x_train, y_train, epochs=3, batch_size=32):  # Minimal memory usage
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        for i in range(0, len(x_train), batch_size):
            x_batch = x_train[i:i + batch_size]
            y_batch = y_train[i:i + batch_size]

            # Create adversarial examples
            adv_x_batch = pgd_attack(model, x_batch, y_batch)

            # Train only on adversarial examples to save memory
            model.train_on_batch(adv_x_batch, y_batch)

# Evaluation function
def evaluate_model(model, x_test, y_test, adversarial=False):
    if adversarial:
        x_test = pgd_attack(model, x_test[:1000], y_test[:1000])  # Use only a subset to save memory
    loss, accuracy = model.evaluate(x_test, y_test[:1000], verbose=0)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    return accuracy

This block compares a standard CNN with an adversarially trained CNN under PGD attack. First, it trains a baseline model on clean CIFAR-10 data and immediately evaluates it on adversarial test examples, revealing how vulnerable a normally trained network is. Next, it builds a fresh model and runs adversarial training: in each minibatch the model’s current weights are used to craft PGD perturbations, and the model is updated on those adversarial inputs, pushing it toward more robust decision boundaries. Finally, both models are evaluated on adversarially perturbed test data and their accuracies are printed. Typically you’ll see the baseline collapse under attack, while the adversarially trained model retains significantly higher accuracy (often at some cost to clean-data accuracy not shown here).

In [5]:
# Train and evaluate models
print("Training baseline model...")
baseline_model = create_simple_cnn()
baseline_model.fit(x_train, y_train, epochs=3, batch_size=32, validation_split=0.1)

print("Evaluating baseline model on adversarial examples...")
baseline_acc = evaluate_model(baseline_model, x_test, y_test, adversarial=True)

print("Training model with adversarial training...")
adv_trained_model = create_simple_cnn()
adversarial_training(adv_trained_model, x_train, y_train, epochs=3)

print("Evaluating adversarially trained model on adversarial examples...")
adv_trained_acc = evaluate_model(adv_trained_model, x_test, y_test, adversarial=True)

print(f"Baseline model accuracy on adversarial examples: {baseline_acc * 100:.2f}%")
print(f"Adversarially trained model accuracy on adversarial examples: {adv_trained_acc * 100:.2f}%")


Training baseline model...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-10-26 01:17:41.080105: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.3284 - loss: 1.8485 - val_accuracy: 0.3750 - val_loss: 1.6225
Epoch 2/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.4684 - loss: 1.4750 - val_accuracy: 0.4340 - val_loss: 1.4799
Epoch 3/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.5231 - loss: 1.3351 - val_accuracy: 0.5080 - val_loss: 1.3405
Evaluating baseline model on adversarial examples...
Accuracy: 8.00%
Training model with adversarial training...
Epoch 1/3
Epoch 2/3
Epoch 3/3
Evaluating adversarially trained model on adversarial examples...
Accuracy: 10.00%
Baseline model accuracy on adversarial examples: 8.00%
Adversarially trained model accuracy on adversarial examples: 10.00%
